In [37]:
# Cell 1: Imports
import time
from pathlib import Path
import numpy as np
import pandas as pd

from time import perf_counter
from prettytable import PrettyTable

from mc_pricer.mc_euro import mc_european
from mc_pricer.mc_asian import mc_asian

In [38]:
def ablation_table(title, pricer, base_kwargs, n_paths_list=(1_000, 10_000, 100_000),
                   methods=("plain", "antithetic", "control_variate")):
    t = PrettyTable()
    t.title = title
    t.field_names = ["Method", "n_paths", "Price", "Std Error", "95% CI Width", "Runtime (s)", "VR"]

    method_label = {
        "plain": "Plain",
        "antithetic": "Antithetic",
        "control_variate": "Control Variate",
    }

    for n in n_paths_list:
        plain_se = None
        ordered = ["plain"] + [m for m in methods if m != "plain"]

        for method in ordered:
            kw = dict(base_kwargs, n_samples=n, method=method)

            # Skip unsupported rows cleanly (no weird N/A row)
            try:
                t0 = perf_counter()
                price, se, ci = pricer(**kw)
                dt = perf_counter() - t0
            except ValueError as e:
                msg = str(e).lower()
                if ("control_variate" in msg and "arithmetic" in msg) or ("method must be" in msg):
                    continue
                raise

            ciw = ci[1] - ci[0]
            if method == "plain":
                plain_se = se
                vr = "1.00x"
            else:
                vr = f"{(plain_se / se) ** 2:.2f}x" if (plain_se and se > 0) else "-"

            t.add_row([
                method_label.get(method, method),
                f"{n:,}",
                f"{price:.4f}",
                f"{se:.6f}",
                f"{ciw:.6f}",
                f"{dt:.4f}",
                vr
            ])

    print(t)


In [39]:
# Cell 3: common params
common = dict(S0=100, K=100, T=1.0, r=0.05, sigma=0.20, option_type="call", seed=42)


In [40]:
# Cell 4: European Call
ablation_table(
    "Ablation: European Call",
    pricer=mc_european,
    base_kwargs=dict(common, option_type="call"),
)


+---------------------------------------------------------------------------------+
|                             Ablation: European Call                             |
+------------+---------+---------+-----------+--------------+-------------+-------+
|   Method   | n_paths |  Price  | Std Error | 95% CI Width | Runtime (s) |   VR  |
+------------+---------+---------+-----------+--------------+-------------+-------+
|   Plain    |  1,000  | 10.5835 |  0.465825 |   1.826035   |    0.0008   | 1.00x |
| Antithetic |  1,000  | 10.4185 |  0.240859 |   0.944168   |    0.0010   | 3.74x |
|   Plain    |  10,000 | 10.4996 |  0.147864 |   0.579626   |    0.0013   | 1.00x |
| Antithetic |  10,000 | 10.4511 |  0.073821 |   0.289378   |    0.0012   | 4.01x |
|   Plain    | 100,000 | 10.5289 |  0.046983 |   0.184173   |    0.0043   | 1.00x |
| Antithetic | 100,000 | 10.4879 |  0.023376 |   0.091632   |    0.0058   | 4.04x |
+------------+---------+---------+-----------+--------------+-------------+-

In [41]:
# Cell 5: European Put
ablation_table(
    "Ablation: European Put",
    pricer=mc_european,
    base_kwargs=dict(common, option_type="put"),
)


+--------------------------------------------------------------------------------+
|                             Ablation: European Put                             |
+------------+---------+--------+-----------+--------------+-------------+-------+
|   Method   | n_paths | Price  | Std Error | 95% CI Width | Runtime (s) |   VR  |
+------------+---------+--------+-----------+--------------+-------------+-------+
|   Plain    |  1,000  | 5.4541 |  0.278538 |   1.091869   |    0.0009   | 1.00x |
| Antithetic |  1,000  | 5.5252 |  0.149767 |   0.587085   |    0.0010   | 3.46x |
|   Plain    |  10,000 | 5.5483 |  0.086463 |   0.338934   |    0.0009   | 1.00x |
| Antithetic |  10,000 | 5.5723 |  0.046831 |   0.183578   |    0.0008   | 3.41x |
|   Plain    | 100,000 | 5.5723 |  0.027365 |   0.107270   |    0.0054   | 1.00x |
| Antithetic | 100,000 | 5.5953 |  0.014867 |   0.058280   |    0.0091   | 3.39x |
+------------+---------+--------+-----------+--------------+-------------+-------+


In [42]:
# Cell 6: Asian Arithmetic Call
ablation_table(
    "Ablation: Asian Arithmetic Call",
    pricer=mc_asian,
    base_kwargs=dict(common, option_type="call", averaging="arithmetic", n_steps=252),
)


+----------------------------------------------------------------------------------------+
|                            Ablation: Asian Arithmetic Call                             |
+-----------------+---------+--------+-----------+--------------+-------------+----------+
|      Method     | n_paths | Price  | Std Error | 95% CI Width | Runtime (s) |    VR    |
+-----------------+---------+--------+-----------+--------------+-------------+----------+
|      Plain      |  1,000  | 5.5723 |  0.244951 |   0.960209   |    0.0158   |  1.00x   |
|    Antithetic   |  1,000  | 5.7118 |  0.122051 |   0.478439   |    0.0263   |  4.03x   |
| Control Variate |  1,000  | 5.7821 |  0.006763 |   0.026512   |    0.0186   | 1311.69x |
|      Plain      |  10,000 | 5.7712 |  0.079171 |   0.310351   |    0.2186   |  1.00x   |
|    Antithetic   |  10,000 | 5.7648 |  0.039004 |   0.152894   |    0.1797   |  4.12x   |
| Control Variate |  10,000 | 5.7830 |  0.002227 |   0.008729   |    0.1371   | 1264.19x |

In [43]:
# Cell 7: Asian Arithmetic Put
ablation_table(
    "Ablation: Asian Arithmetic Put",
    pricer=mc_asian,
    base_kwargs=dict(common, option_type="put", averaging="arithmetic", n_steps=252),
)


+----------------------------------------------------------------------------------------+
|                             Ablation: Asian Arithmetic Put                             |
+-----------------+---------+--------+-----------+--------------+-------------+----------+
|      Method     | n_paths | Price  | Std Error | 95% CI Width | Runtime (s) |    VR    |
+-----------------+---------+--------+-----------+--------------+-------------+----------+
|      Plain      |  1,000  | 3.4333 |  0.168111 |   0.658996   |    0.0157   |  1.00x   |
|    Antithetic   |  1,000  | 3.3076 |  0.090174 |   0.353481   |    0.0241   |  3.48x   |
| Control Variate |  1,000  | 3.3577 |  0.003803 |   0.014907   |    0.0195   | 1954.29x |
|      Plain      |  10,000 | 3.3441 |  0.052654 |   0.206403   |    0.1259   |  1.00x   |
|    Antithetic   |  10,000 | 3.3415 |  0.028551 |   0.111921   |    0.1746   |  3.40x   |
| Control Variate |  10,000 | 3.3537 |  0.001338 |   0.005244   |    0.1317   | 1549.18x |

In [44]:
# Cell 8: Asian Geometric Call
ablation_table(
    "Ablation: Asian Geometric Call",
    pricer=mc_asian,
    base_kwargs=dict(common, option_type="call", averaging="geometric", n_steps=252),
)


+--------------------------------------------------------------------------------+
|                         Ablation: Asian Geometric Call                         |
+------------+---------+--------+-----------+--------------+-------------+-------+
|   Method   | n_paths | Price  | Std Error | 95% CI Width | Runtime (s) |   VR  |
+------------+---------+--------+-----------+--------------+-------------+-------+
|   Plain    |  1,000  | 5.3628 |  0.236475 |   0.926980   |    0.0302   | 1.00x |
| Antithetic |  1,000  | 5.5004 |  0.118292 |   0.463705   |    0.0357   | 4.00x |
|   Plain    |  10,000 | 5.5540 |  0.076442 |   0.299654   |    0.1732   | 1.00x |
| Antithetic |  10,000 | 5.5478 |  0.037748 |   0.147972   |    0.2071   | 4.10x |
|   Plain    | 100,000 | 5.5281 |  0.024402 |   0.095657   |    1.2888   | 1.00x |
| Antithetic | 100,000 | 5.5643 |  0.012009 |   0.047076   |    1.9126   | 4.13x |
+------------+---------+--------+-----------+--------------+-------------+-------+


In [45]:
# Cell 9: Asian Geometric Put
ablation_table(
    "Ablation: Asian Geometric Put",
    pricer=mc_asian,
    base_kwargs=dict(common, option_type="put", averaging="geometric", n_steps=252),
)

+--------------------------------------------------------------------------------+
|                         Ablation: Asian Geometric Put                          |
+------------+---------+--------+-----------+--------------+-------------+-------+
|   Method   | n_paths | Price  | Std Error | 95% CI Width | Runtime (s) |   VR  |
+------------+---------+--------+-----------+--------------+-------------+-------+
|   Plain    |  1,000  | 3.5501 |  0.172578 |   0.676504   |    0.0177   | 1.00x |
| Antithetic |  1,000  | 3.4229 |  0.092128 |   0.361143   |    0.0215   | 3.51x |
|   Plain    |  10,000 | 3.4625 |  0.054103 |   0.212083   |    0.1302   | 1.00x |
| Antithetic |  10,000 | 3.4601 |  0.029178 |   0.114377   |    0.1922   | 3.44x |
|   Plain    | 100,000 | 3.4987 |  0.017117 |   0.067100   |    1.3046   | 1.00x |
| Antithetic | 100,000 | 3.4717 |  0.009265 |   0.036318   |    1.9371   | 3.41x |
+------------+---------+--------+-----------+--------------+-------------+-------+
